# Пункт 2.2. Источники данных и их первичное изучение

**Тема курсовой работы:** «Исследование геолокационных данных горожан для обнаружения паттернов передвижения и активности».

В практической части объект исследования представлен обезличенными поездками NYC Yellow Taxi за 2025 год. Географическая единица анализа — официальная Taxi Zone. Цель этого ноутбука — проверить выбранные источники, описать их структуру и выбрать числовые и категориальные признаки для дальнейшего анализа.

## Используемые источники

1. **Реляционный набор:** NYC TLC Yellow Taxi Trip Records — основной объект исследования.
2. **Иерархический XML-набор:** NYC 311 Service Requests — обращения горожан как фактор состояния городской среды.
3. **JSON API:** NYPD Motor Vehicle Collisions — ДТП как фактор изменения условий движения.
4. **Неструктурированные данные:** архив пресс-релизов NYC DOT за 2025 год — официальные тексты о дорожных и транспортных событиях.

Единый пространственный контекст — Нью-Йорк. Такси объединяется с внешними источниками по Taxi Zone и времени: ДТП и обращения 311 относятся к зоне автоматически по координатам, а поездки уже содержат идентификаторы зон.

In [1]:
import sys
!{sys.executable} -m pip install beautifulsoup4

In [2]:
import sys
!{sys.executable} -m pip install --upgrade --force-reinstall "numpy==1.26.4" "pandas==2.2.3" "pyarrow==18.1.0" "geopandas==1.0.1" "shapely==2.0.6" requests

  Using cached numpy-1.26.4-cp312-cp312-win_amd64.whl.metadata (61 kB)
  Using cached pandas-2.2.3-cp312-cp312-win_amd64.whl.metadata (19 kB)
  Using cached pyarrow-18.1.0-cp312-cp312-win_amd64.whl.metadata (3.4 kB)
  Using cached geopandas-1.0.1-py3-none-any.whl.metadata (2.2 kB)
  Using cached shapely-2.0.6-cp312-cp312-win_amd64.whl.metadata (7.2 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl.metadata (8.4 kB)
  Using cached pytz-2026.5-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2026.5-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached pyogrio-0.13.0-cp311-abi3-win_amd64.whl.metadata (6.0 kB)
  Using cached packaging-26.3-py3-none-any.whl.metadata (3.5 kB)
  Using cached pyproj-3.8.0-cp312-cp312-win_amd64.whl.metadata (40 kB)
  Using cached charset_normalizer-3.5.2-cp312-cp312-win_amd64.whl.metadata (47 kB)
  Using cached idna-3.20-py3-none-any.whl.metadata (7.2 kB)
  Using cach

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.32.0 requires packaging<24,>=16.8, but you have packaging 26.3 which is incompatible.


In [3]:
# Проверка версий
import numpy as np
import pandas as pd
import geopandas as gpd
import shapely

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("GeoPandas:", gpd.__version__)
print("Shapely:", shapely.__version__)

NumPy: 1.26.4
Pandas: 2.2.3
GeoPandas: 1.0.1
Shapely: 2.0.6


In [4]:
from pathlib import Path
from urllib.parse import urljoin
from xml.etree import ElementTree as ET
import re

import geopandas as gpd
import pandas as pd
import requests
from bs4 import BeautifulSoup
from IPython.display import display

# Ноутбук можно запускать из папки проекта или из папки notebooks.
start_dir = Path.cwd().resolve()
PROJECT_DIR = next(folder for folder in [start_dir, *start_dir.parents] if (folder / 'data').exists())
RAW_DIR = PROJECT_DIR / 'data' / 'raw'
PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f'Папка проекта: {PROJECT_DIR}')

Папка проекта: C:\Users\User\Documents\3 курс\ТОБД\GeoData


In [5]:
def make_profile(dataframe: pd.DataFrame) -> pd.DataFrame:
    """Создаёт компактный профиль столбцов для первичного изучения источника."""
    return pd.DataFrame({
        'column': dataframe.columns,
        'dtype': dataframe.dtypes.astype(str).values,
        'missing_count': dataframe.isna().sum().values,
        'example_value': [
            dataframe[column].dropna().iloc[0] if dataframe[column].notna().any() else None
            for column in dataframe.columns
        ],
    })

def save_profile(dataframe: pd.DataFrame, name: str) -> pd.DataFrame:
    profile = make_profile(dataframe)
    profile.to_csv(PROCESSED_DIR / f'{name}_columns.csv', index=False, encoding='utf-8-sig')
    return profile

## 1. Основной реляционный набор: поездки Yellow Taxi

Каждая строка описывает одну завершённую поездку. `PULocationID` и `DOLocationID` выглядят как числа, но по смыслу это **категориальные идентификаторы зон**.

In [6]:
taxi_path = RAW_DIR / 'yellow_tripdata_2025-01.parquet'
if not taxi_path.exists():
    raise FileNotFoundError(f'Не найден файл {taxi_path}. Поместите его в data/raw/.')

taxi = pd.read_parquet(taxi_path)
print(f'NYC Yellow Taxi: {taxi.shape[0]:,} строк, {taxi.shape[1]} столбцов')
taxi_profile = save_profile(taxi, 'taxi')
display(taxi_profile)
display(taxi.head(3))

NYC Yellow Taxi: 3,475,226 строк, 20 столбцов


,column,dtype,missing_count,example_value
0,VendorID,int32,0,1
1,tpep_pickup_datetime,datetime64[us],0,2025-01-01 00:18:38
2,tpep_dropoff_datetime,datetime64[us],0,2025-01-01 00:26:59
3,passenger_count,float64,540149,1.0
4,trip_distance,float64,0,1.6
5,RatecodeID,float64,540149,1.0
6,store_and_fwd_flag,object,540149,N
7,PULocationID,int32,0,229
8,DOLocationID,int32,0,237
9,payment_type,int64,0,1


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.6,1.0,N,229,237,1,10.0,3.5,0.5,3.00,0.0,1.0,18.00,2.5,0.0,0.0
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.5,1.0,N,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.6,1.0,N,141,141,1,5.1,3.5,0.5,2.00,0.0,1.0,12.10,2.5,0.0,0.0


- **Номинальная ID:** `PULocationID`, `DOLocationID`.
- **Категориальная:** `VendorID`, `RatecodeID`, `payment_type`.
- **Дихотомическая:** `store_and_fwd_flag`.
- **Интервальная:** `tpep_pickup_datetime`, `tpep_dropoff_datetime`.
- **Абсолютная:** `passenger_count`.
- **Отношений:** `trip_distance`, `fare_amount`, `extra`, `mta_tax`, `tip_amount`, `tolls_amount`, `improvement_surcharge`, `total_amount`, `congestion_surcharge`, `Airport_fee`, `cbd_congestion_fee`.

`PULocationID` и `DOLocationID` хранятся как целые числа, но среднее или разность идентификаторов зон не имеют смысла. Их используют для группировки и объединения данных.

## 2. Пространственный справочник: официальные Taxi Zones

Shapefile состоит из нескольких файлов, поэтому они должны лежать рядом. `LocationID` является ключом, который связывает границы и названия зон с `PULocationID` и `DOLocationID` поездок.

In [7]:
shape_files = list((RAW_DIR / 'taxi_zones').rglob('*.shp'))
if not shape_files:
    raise FileNotFoundError('В data/raw/taxi_zones не найден файл .shp.')

zones = gpd.read_file(shape_files[0])
print(f'NYC Taxi Zones: {zones.shape[0]} строк, {zones.shape[1]} столбцов')
print(f'Система координат: {zones.crs}')
zones_profile = save_profile(zones.drop(columns='geometry'), 'taxi_zones')
display(zones_profile)
display(zones[['LocationID', 'zone', 'borough', 'geometry']].head(3))

NYC Taxi Zones: 263 строк, 7 столбцов
Система координат: EPSG:2263


,column,dtype,missing_count,example_value
0,OBJECTID,int32,0,1
1,Shape_Leng,float64,0,0.116357
2,Shape_Area,float64,0,0.000782
3,zone,object,0,Newark Airport
4,LocationID,int32,0,1
5,borough,object,0,EWR


,LocationID,zone,borough,geometry
0,1,Newark Airport,EWR,"POLYGON ((933100.918 192536.086, 933091.011 19..."
1,2,Jamaica Bay,Queens,"MULTIPOLYGON (((1033269.244 172126.008, 103343..."
2,3,Allerton/Pelham Gardens,Bronx,"POLYGON ((1026308.77 256767.698, 1026495.593 2..."


- **Номинальная ID:** `OBJECTID`, `LocationID`.
- **Категориальная:** `zone`, `borough`.
- **Контентная:** `geometry`.
- **Отношений:** `Shape_Leng`, `Shape_Area`.

`geometry` представляет полигон зоны и является сложным геопространственным объектом, а не обычным числом или категорией.

## 3. Источник JSON API: ДТП NYPD

Запрашивается ограниченная выборка в размере 1 000 записей за 2025 год. Этого достаточно для первичного изучения и не требует скачивания большого файла вручную.

In [8]:
COLLISIONS_URL = 'https://data.cityofnewyork.us/resource/h9gi-nx95.json'
collisions_params = {
    '$limit': 1000,
    '$where': "crash_date between '2025-01-01T00:00:00' and '2025-12-31T23:59:59'",
}
collisions_response = requests.get(COLLISIONS_URL, params=collisions_params, timeout=60)
collisions_response.raise_for_status()
collisions = pd.DataFrame(collisions_response.json())

print(f'NYPD collisions: {collisions.shape[0]:,} строк, {collisions.shape[1]} столбцов')
display(save_profile(collisions, 'collisions_json'))

NYPD collisions: 1,000 строк, 29 столбцов


,column,dtype,missing_count,example_value
0,crash_date,object,0,2025-01-01T00:00:00.000
1,crash_time,object,0,11:20
2,borough,object,186,BROOKLYN
3,zip_code,object,186,11213
4,latitude,object,16,40.66658
5,longitude,object,16,-73.939674
6,location,object,16,"{'latitude': '40.66658', 'longitude': '-73.939..."
7,on_street_name,object,353,ALBANY AVE
8,off_street_name,object,447,CARROLL ST
9,number_of_persons_injured,object,0,0


- **Номинальная ID:** `collision_id`, `zip_code`.
- **Категориальная:** `borough`, `contributing_factor_vehicle_1`, `vehicle_type_code1`, `contributing_factor_vehicle_2`, `vehicle_type_code2`, `contributing_factor_vehicle_3`, `contributing_factor_vehicle_4`, `vehicle_type_code_3`, `vehicle_type_code_4`, `contributing_factor_vehicle_5`, `vehicle_type_code_5`.
- **Контентная:** `location`, `on_street_name`, `off_street_name`, `cross_street_name`.
- **Интервальная:** `crash_date`, `crash_time`, `latitude`, `longitude`.
- **Абсолютная:** `number_of_persons_injured`, `number_of_persons_killed`, `number_of_pedestrians_injured`, `number_of_pedestrians_killed`, `number_of_cyclist_injured`, `number_of_cyclist_killed`, `number_of_motorist_injured`, `number_of_motorist_killed`.

Широта и долгота имеют равный шаг, но их нулевые точки условны, поэтому это интервальная, а не шкала отношений. Поле `location` передаётся API как вложенный JSON-объект.

## 4. Иерархический XML-набор: обращения NYC 311

XML получаем напрямую от NYC Open Data. Файл сохраняется локально, но не добавляется в GitHub. XML имеет вложенную структуру: корневой элемент содержит записи обращений, а каждая запись — отдельные поля.

In [9]:
SERVICE_311_URL = 'https://data.cityofnewyork.us/resource/erm2-nwe9.xml'
service_311_params = {
    '$limit': 1000,
    '$where': "created_date between '2025-01-01T00:00:00' and '2025-12-31T23:59:59'",
}
service_311_response = requests.get(SERVICE_311_URL, params=service_311_params, timeout=60)
service_311_response.raise_for_status()
xml_path = RAW_DIR / 'nyc_311_2025_sample.xml'
xml_path.write_bytes(service_311_response.content)

root = ET.fromstring(service_311_response.content)
xml_rows = []
for row in root.findall('.//row'):
    xml_rows.append({child.tag: child.text for child in row})
service_311 = pd.DataFrame(xml_rows)

print(f'NYC 311 XML: {service_311.shape[0]:,} строк, {service_311.shape[1]} столбцов')
print(f'Локальная копия XML: {xml_path}')
display(save_profile(service_311, 'service_311_xml'))
display(service_311.head(3))

NYC 311 XML: 1,001 строк, 42 столбцов
Локальная копия XML: C:\Users\User\Documents\3 курс\ТОБД\GeoData\data\raw\nyc_311_2025_sample.xml


,column,dtype,missing_count,example_value
0,row,float64,1001,None
1,unique_key,object,1,67351762
2,created_date,object,1,2025-12-31T23:59:28
3,closed_date,object,8,2026-01-01T00:40:32
4,agency,object,1,NYPD
5,agency_name,object,1,New York City Police Department
6,complaint_type,object,1,Noise - Residential
7,descriptor,object,1,Loud Music/Party
8,location_type,object,45,Residential Building/House
9,incident_zip,object,5,10029


,row,unique_key,created_date,closed_date,agency,agency_name,complaint_type,descriptor,location_type,incident_zip,...,latitude,longitude,location,descriptor_2,taxi_pick_up_location,vehicle_type,facility_type,bridge_highway_name,bridge_highway_segment,bridge_highway_direction
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,67351762,2025-12-31T23:59:28,2026-01-01T00:40:32,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,Residential Building/House,10029,...,40.792140506423976,-73.9500970728466,POINT (-73.950097072847 40.792140506424),NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,67344624,2025-12-31T23:59:23,2026-01-01T01:03:42,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,Residential Building/House,10031,...,40.82513729000985,-73.94944723723135,POINT (-73.949447237231 40.82513729001),NaN,NaN,NaN,NaN,NaN,NaN,NaN


- **Номинальная ID:** `row`, `unique_key`, `incident_zip`, `community_board`, `council_district`, `police_precinct`, `bbl`.
- **Категориальная:** `agency`, `agency_name`, `complaint_type`, `descriptor`, `location_type`, `address_type`, `city`, `status`, `borough`, `open_data_channel_type`, `park_facility_name`, `park_borough`, `descriptor_2`, `vehicle_type`, `facility_type`, `bridge_highway_direction`.
- **Контентная:** `incident_address`, `street_name`, `cross_street_1`, `cross_street_2`, `intersection_street_1`, `intersection_street_2`, `landmark`, `resolution_description`, `location`, `taxi_pick_up_location`, `bridge_highway_name`, `bridge_highway_segment`.
- **Интервальная:** `created_date`, `closed_date`, `resolution_action_updated_date`, `x_coordinate_state_plane`, `y_coordinate_state_plane`, `latitude`, `longitude`.

`row` является технической строкой XML и в аналитических моделях не используется. `unique_key` также применяют только как технический идентификатор. Для 311 дополнительно рассчитываются числовые признаки: час создания, длительность обработки и число обращений в зоне за час. Последний показатель относится к абсолютной шкале.


## 5. Неструктурированные текстовые данные: пресс-релизы NYC DOT

Используется официальный архив пресс-релизов NYC Department of Transportation за 2025 год. Каждый объект источника — HTML-страница с неструктурированным текстом релиза. В текстах встречаются дорожные работы, изменения организации движения, безопасность, улицы и районы Нью-Йорка.

In [10]:
DOT_ARCHIVE_URL = 'https://www.nyc.gov/html/dot/html/pr2025/prindex-2025.shtml'

# NYC.gov может блокировать запросы без заголовка браузера.
dot_session = requests.Session()
dot_session.headers.update({
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 Chrome/131.0 Safari/537.36'
    ),
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.9',
})

archive_response = dot_session.get(DOT_ARCHIVE_URL, timeout=60)
archive_response.raise_for_status()
archive_soup = BeautifulSoup(archive_response.text, 'html.parser')

# Берём первые 10 страниц релизов из официального архива 2025 года.
article_links = []

for link in archive_soup.select('a[href]'):
    url = urljoin(DOT_ARCHIVE_URL, link['href'])
    title = link.get_text(' ', strip=True)

    if '/pr2025/' in url and title and url not in [item['url'] for item in article_links]:
        article_links.append({'title': title, 'url': url})

transport_words = re.compile(
    r'\b(traffic|street|road|bridge|transit|bike|safety|parking|construction)\b',
    re.I
)

borough_words = ['Bronx', 'Brooklyn', 'Manhattan', 'Queens', 'Staten Island']

topic_patterns = {
    'construction': r'construction|resurfacing|repair|work zone',
    'street_safety': r'safety|vision zero|pedestrian|intersection',
    'cycling': r'bike|bicycle|cycling',
    'traffic_management': r'traffic|parking|congestion|street closure',
}

dot_rows = []

for item in article_links[:10]:
    article_response = dot_session.get(item['url'], timeout=60)
    article_response.raise_for_status()

    article_text = BeautifulSoup(
        article_response.text,
        'html.parser'
    ).get_text(' ', strip=True)

    date_match = re.search(r'[A-Z][a-z]+\s+\d{1,2},\s+2025', article_text)

    borough = next(
        (
            name for name in borough_words
            if re.search(rf'\b{name}\b', article_text, re.I)
        ),
        'not_specified'
    )

    topic = next(
        (
            name for name, pattern in topic_patterns.items()
            if re.search(pattern, article_text, re.I)
        ),
        'other'
    )

    published_date = date_match.group(0) if date_match else None

    dot_rows.append({
        'source': 'NYC DOT',
        'document_type': 'press_release',
        'title': item['title'],
        'url': item['url'],
        'published_date': published_date,
        'publication_month': pd.to_datetime(published_date).month if published_date else None,
        'borough_mentioned': borough,
        'topic': topic,
        'title_length': len(item['title']),
        'text_length': len(article_text),
        'transport_word_count': len(transport_words.findall(article_text)),
    })

dot_text = pd.DataFrame(dot_rows)

print(
    f'NYC DOT press releases: '
    f'{dot_text.shape[0]} материалов, {dot_text.shape[1]} полей'
)

display(save_profile(dot_text, 'nyc_dot_text'))

display(
    dot_text[
        [
            'published_date',
            'borough_mentioned',
            'title',
            'text_length',
            'transport_word_count'
        ]
    ]
)

NYC DOT press releases: 10 материалов, 11 полей


,column,dtype,missing_count,example_value
0,source,object,0,NYC DOT
1,document_type,object,0,press_release
2,title,object,0,Skip to main content
3,url,object,0,https://www.nyc.gov/html/dot/html/pr2025/prind...
4,published_date,object,0,"December 28, 2025"
5,publication_month,int64,0,12
6,borough_mentioned,object,0,Bronx
7,topic,object,0,construction
8,title_length,int64,0,20
9,text_length,int64,0,21876


,published_date,borough_mentioned,title,text_length,transport_word_count
0,"December 28, 2025",Bronx,Skip to main content,21876,114
1,"December 28, 2025",Staten Island,Suspension of Alternate\r\n Side Parkin...,1647,24
2,"December 26, 2025",Manhattan,NYC DOT Names New Washington Heights Plaza Ali...,6489,22
3,"December 26, 2025",Staten Island,Suspension of Alternate\r\n Side Parkin...,1649,24
4,"December 23, 2025",Staten Island,Vision Zero: New York City Will Aggressively\r...,5734,23
5,"December 19, 2025",Bronx,NYC DOT Releases Harlem River Greenway\r\n ...,9251,45
6,"December 18, 2025",Manhattan,NYC DOT Commissioner Ydanis\r\n Rodrigu...,8511,25
7,"December 17, 2025",Brooklyn,NYC DOT Releases Limited Batch of Fifth\r\n ...,7395,43
8,"December 16, 2025",Bronx,NYC DOT Celebrates Four Years of Historic Prog...,12292,60
9,"December 16, 2025",Staten Island,Suspension of Alternate Side\r\n Parkin...,1686,24


- **Категориальная:** `source`, `document_type`, `borough_mentioned`, `topic`.
- **Контентная:** `title`, `url`.
- **Порядковая:** `publication_month`.
- **Интервальная:** `published_date`.
- **Абсолютная:** `title_length`, `text_length`, `transport_word_count`.

`title_length` и `text_length` измеряют число символов, а `transport_word_count` — число найденных транспортных ключевых слов. Это счётные величины с фиксированной единицей, поэтому они относятся к абсолютной шкале.

In [11]:
scale_catalog = pd.DataFrame([
    {'source': 'Yellow Taxi', 'feature': 'PULocationID, DOLocationID', 'scale': 'номинальная ID', 'meaning': 'идентификаторы зон; соединение и частоты, без среднего'},
    {'source': 'Yellow Taxi', 'feature': 'VendorID, RatecodeID, payment_type', 'scale': 'категориальная', 'meaning': 'конечные коды без естественного порядка'},
    {'source': 'Yellow Taxi', 'feature': 'store_and_fwd_flag', 'scale': 'дихотомическая', 'meaning': 'два состояния Y/N; допустима доля значений Y'},
    {'source': 'Yellow Taxi', 'feature': 'tpep_pickup_datetime', 'scale': 'интервальная', 'meaning': 'дата и время; осмысленны разности и группировка по часу'},
    {'source': 'Yellow Taxi', 'feature': 'trip_distance, fare_amount, total_amount, trip_duration', 'scale': 'отношений', 'meaning': 'есть естественный ноль; допустимы отношения и среднее'},
    {'source': 'Yellow Taxi', 'feature': 'passenger_count, trip_count', 'scale': 'абсолютная', 'meaning': 'счёт объектов с неделимой единицей'},
    {'source': 'NYPD Collisions', 'feature': 'collision_id', 'scale': 'номинальная ID', 'meaning': 'технический идентификатор ДТП'},
    {'source': 'NYPD Collisions', 'feature': 'borough, vehicle_type_code1, contributing_factor_vehicle_1', 'scale': 'категориальная', 'meaning': 'категории без порядка'},
    {'source': 'NYPD Collisions', 'feature': 'injury_severity', 'scale': 'порядковая', 'meaning': 'рассчитываемая градация: нет пострадавших, есть пострадавшие, есть погибшие'},
    {'source': 'NYPD Collisions', 'feature': 'number_of_persons_injured, number_of_persons_killed', 'scale': 'абсолютная', 'meaning': 'счёт людей'},
    {'source': 'NYC 311 XML', 'feature': 'unique_key', 'scale': 'номинальная ID', 'meaning': 'технический идентификатор обращения'},
    {'source': 'NYC 311 XML', 'feature': 'complaint_type, agency, borough, status', 'scale': 'категориальная', 'meaning': 'конечные наборы категорий'},
    {'source': 'NYC 311 XML', 'feature': 'created_date, closed_date', 'scale': 'интервальная', 'meaning': 'время создания и закрытия; по разности получается длительность'},
    {'source': 'NYC 311 XML', 'feature': 'processing_duration, request_count', 'scale': 'отношений / абсолютная', 'meaning': 'длительность имеет естественный ноль, число обращений является счётом'},
    {'source': 'NYC DOT text', 'feature': 'title, article_text, url', 'scale': 'контентная', 'meaning': 'неструктурированный текст и ссылка'},
    {'source': 'NYC DOT text', 'feature': 'topic, borough_mentioned, publication_month', 'scale': 'категориальная', 'meaning': 'выделенные из текста категории'},
    {'source': 'NYC DOT text', 'feature': 'title_length, text_length, transport_word_count', 'scale': 'абсолютная', 'meaning': 'счёт символов или слов'},
])
display(scale_catalog)

,source,feature,scale,meaning
0,Yellow Taxi,"PULocationID, DOLocationID",номинальная ID,"идентификаторы зон; соединение и частоты, без ..."
1,Yellow Taxi,"VendorID, RatecodeID, payment_type",категориальная,конечные коды без естественного порядка
2,Yellow Taxi,store_and_fwd_flag,дихотомическая,два состояния Y/N; допустима доля значений Y
3,Yellow Taxi,tpep_pickup_datetime,интервальная,дата и время; осмысленны разности и группировк...
4,Yellow Taxi,"trip_distance, fare_amount, total_amount, trip...",отношений,есть естественный ноль; допустимы отношения и ...
5,Yellow Taxi,"passenger_count, trip_count",абсолютная,счёт объектов с неделимой единицей
6,NYPD Collisions,collision_id,номинальная ID,технический идентификатор ДТП
7,NYPD Collisions,"borough, vehicle_type_code1, contributing_fact...",категориальная,категории без порядка
8,NYPD Collisions,injury_severity,порядковая,"рассчитываемая градация: нет пострадавших, ест..."
9,NYPD Collisions,"number_of_persons_injured, number_of_persons_k...",абсолютная,счёт людей


## Вывод по пункту 2.2

Выбраны четыре источника требуемых типов. Основной набор — поездки Yellow Taxi. Taxi Zone связывает поездки с пространственными справочниками, ДТП и обращениями 311. В дальнейшем будет создана аналитическая таблица, где одна строка описывает показатели одной Taxi Zone за один час конкретной даты: количество поездок, средняя дистанция и стоимость, число ДТП и число обращений 311. Тексты пресс-релизов NYC DOT будут использоваться как дополнительный городской контекст на уровне borough и даты.